# Home Loan Default Risk — Exploratory Data Analysis
**Project:** PRCP-1006 Home Loan Default – Risk Management · **Task 1:** Data analysis report

**Business problem.** Home Credit lends to people with little or no credit history. It wants to predict, at the moment of application, whether an applicant will have **payment difficulties** (`TARGET = 1`), using the application itself plus the applicant's history with other lenders (credit bureau) and with Home Credit.

**What this notebook does**
1. Loads all 7 tables and checks their size, keys and how they link.
2. Audits data quality: missing values, placeholder codes, anomalies, outliers, key integrity.
3. Analyses the target (default rate and class imbalance).
4. Explores the main application table: who defaults, and which features separate defaulters from non-defaulters.
5. Explores each history table, aggregates it to one row per applicant and measures its link with default.
6. Ranks the strongest signals and summarises findings and recommendations for the modelling task.
7. Writes a stand-alone **written report** (`eda_outputs/Home_Loan_EDA_Report.html`) from the numbers computed here.

> Run it top to bottom on the **full** Kaggle dataset (*Home Credit Default Risk*). All numbers, charts and the written report are computed from the data, so they reflect whatever data you run it on.

## 1. Setup
Needs `pandas`, `numpy`, `matplotlib`, `scikit-learn` and `jinja2` (all preinstalled on Kaggle). Locally: `pip install pandas numpy matplotlib scikit-learn jinja2`.

In [ ]:
import os, gc, base64, warnings, datetime
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# ---- Where the CSV files are. The first folder that contains application_train.csv is used.
CANDIDATE_DIRS = [
    "/kaggle/input/home-credit-default-risk",   # Kaggle notebook with the competition data attached
    "data", "data/raw", "../data", ".",
]
DATA_DIR = next((Path(d) for d in CANDIDATE_DIRS if (Path(d) / "application_train.csv").exists()), None)
assert DATA_DIR is not None, "application_train.csv not found - put the 7 CSV files in ./data or edit CANDIDATE_DIRS"
print("Reading data from:", DATA_DIR.resolve())

OUT_DIR = Path("eda_outputs"); FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ---- Chart style: one consistent, readable look for every figure
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"     # repaid / defaulted / extra series
INK, INK2, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "axes.titlecolor": INK, "axes.titlesize": 13,
    "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.labelsize": 10.5,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 9.5, "ytick.labelsize": 9.5,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
    "legend.fontsize": 9.5, "figure.dpi": 110, "font.size": 10,
})

FIGS = []          # (file, caption) - collected for the written report
REPORT = {}        # numbers used in the written report
FINDINGS = {}      # section -> list of finding sentences

def save(fig, name, caption):
    # Save a figure, register it for the report and show it.
    path = FIG_DIR / f"{len(FIGS)+1:02d}_{name}.png"
    fig.savefig(path, dpi=130, bbox_inches="tight")
    FIGS.append((path, caption))
    plt.show()

def note(section, text):
    # Record a finding for the report and print it under the chart.
    FINDINGS.setdefault(section, []).append(text)
    display(Markdown(f"**Finding:** {text}"))

pct = lambda x: f"{x:.1%}"

def more_less(a, b, tol=1.1):
    # "higher than" / "lower than" / "about the same as" for rate a compared with rate b
    return "about the same as" if max(a, b) / max(min(a, b), 1e-9) < tol else ("higher than" if a > b else "lower than")

## 2. Load the data
The full tables are large (tens of millions of rows in the monthly history tables). To keep memory down, text columns are stored as `category` and numbers are down-cast to the smallest type that holds them.

> **Tip:** if your laptop runs out of memory, run this notebook on Kaggle (free, about 30 GB RAM): *New Notebook → Add data → Home Credit Default Risk*. The path is detected automatically.

In [ ]:
def shrink(df):
    # Down-cast numbers and turn text columns into categories to save memory.
    for c in df.columns:
        s = df[c]
        if pd.api.types.is_float_dtype(s):
            df[c] = pd.to_numeric(s, downcast="float")
        elif pd.api.types.is_integer_dtype(s):
            df[c] = pd.to_numeric(s, downcast="integer")
        elif not pd.api.types.is_numeric_dtype(s):
            df[c] = s.astype("category")
    return df

TABLES = ["application_train", "bureau", "bureau_balance", "previous_application",
          "POS_CASH_balance", "credit_card_balance", "installments_payments"]
data = {}
for t in TABLES:
    data[t] = shrink(pd.read_csv(DATA_DIR / f"{t}.csv"))
    print(f"{t:<24}{data[t].shape[0]:>12,} rows {data[t].shape[1]:>5} cols "
          f"{data[t].memory_usage(deep=True).sum()/1e6:>9,.0f} MB")

app, bureau, bb, prev = data["application_train"], data["bureau"], data["bureau_balance"], data["previous_application"]
pos, cc, inst = data["POS_CASH_balance"], data["credit_card_balance"], data["installments_payments"]

## 3. Dataset overview and structure

The data is relational: one main table with one row per applicant, plus six history tables with many rows per applicant.

```
application_train  (1 row per applicant, has TARGET)              key SK_ID_CURR
├── bureau                (loans at other lenders)                  key SK_ID_BUREAU
│     └── bureau_balance  (monthly status of each bureau loan)
└── previous_application  (past Home Credit applications)           key SK_ID_PREV
      ├── POS_CASH_balance       (monthly snapshots, POS / cash loans)
      ├── credit_card_balance    (monthly snapshots, credit cards)
      └── installments_payments  (each instalment: due vs paid)
```

| Table | One row is… | Joins to |
|---|---|---|
| application_train | one loan application | — |
| bureau | one loan at another lender | `SK_ID_CURR` |
| bureau_balance | one bureau loan in one month | `SK_ID_BUREAU` |
| previous_application | one earlier Home Credit application | `SK_ID_CURR` |
| POS_CASH_balance | one past POS/cash loan in one month | `SK_ID_PREV`, `SK_ID_CURR` |
| credit_card_balance | one past credit card in one month | `SK_ID_PREV`, `SK_ID_CURR` |
| installments_payments | one instalment of a past loan | `SK_ID_PREV`, `SK_ID_CURR` |

In [ ]:
KEYS = {"application_train": "SK_ID_CURR", "bureau": "SK_ID_BUREAU", "bureau_balance": "SK_ID_BUREAU",
        "previous_application": "SK_ID_PREV", "POS_CASH_balance": "SK_ID_PREV",
        "credit_card_balance": "SK_ID_PREV", "installments_payments": "SK_ID_PREV"}
rows = []
for t, df in data.items():
    num = sum(pd.api.types.is_numeric_dtype(df[c]) for c in df.columns)
    rows.append({"table": t, "rows": len(df), "columns": df.shape[1], "numeric cols": num,
                 "text cols": df.shape[1] - num, "key": KEYS[t], "unique keys": df[KEYS[t]].nunique(),
                 "rows per applicant": round(len(df) / df["SK_ID_CURR"].nunique(), 1) if "SK_ID_CURR" in df else None,
                 "missing cells %": round(df.isna().mean().mean() * 100, 1),
                 "duplicate rows": f"{int(df.duplicated().sum()):,}" if len(df) < 5_000_000 else "not checked (large)"})
overview = pd.DataFrame(rows).set_index("table")
REPORT["overview"] = overview
display(overview)
REPORT["n_app"] = len(app); REPORT["n_cols_app"] = app.shape[1]
REPORT["total_rows"] = int(overview["rows"].sum())

In [ ]:
# Key integrity: how many applicants have history in each table, and are there orphan keys?
ids = set(app["SK_ID_CURR"])
cover = []
for t in ["bureau", "previous_application", "POS_CASH_balance", "credit_card_balance", "installments_payments"]:
    child = set(data[t]["SK_ID_CURR"])
    cover.append({"table": t, "applicants with history %": round(len(ids & child) / len(ids) * 100, 1),
                  "IDs not in application_train %": round(len(child - ids) / max(len(child), 1) * 100, 1)})
bb_link = bb["SK_ID_BUREAU"].isin(bureau["SK_ID_BUREAU"]).mean()
coverage = pd.DataFrame(cover).set_index("table")
REPORT["coverage"] = coverage
display(coverage)
print(f"bureau_balance rows whose SK_ID_BUREAU exists in bureau: {bb_link:.1%}")

**Reading the table above.** IDs that are *not* in `application_train` belong to applicants in the separate test set (`application_test.csv`), so they are expected, not an error. Applicants with **no** history in a table are not missing data either: it means they had no such loans, which is itself informative (thin credit file).

In [ ]:
no_bureau = 1 - coverage.loc["bureau", "applicants with history %"] / 100
REPORT["no_bureau"] = no_bureau
note("Data overview", f"The data has {len(TABLES)} tables and {REPORT['total_rows']:,} rows in total. "
     f"The main table has {REPORT['n_app']:,} applicants and {REPORT['n_cols_app']} columns. "
     f"{pct(no_bureau)} of applicants have no record at the credit bureau, so history-based features "
     f"will be missing for them and the model must handle that.")

## 4. Data quality audit
### 4.1 Missing values

In [ ]:
miss = (app.isna().mean() * 100).sort_values(ascending=False)
miss = miss[miss > 0]
REPORT["miss_cols"] = len(miss); REPORT["miss_over50"] = int((miss > 50).sum())
print(f"{len(miss)} of {app.shape[1]} columns have missing values; {(miss > 50).sum()} are more than 50% empty.")

top = miss.head(30)[::-1]
fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(top.index, top.values, color=BLUE, height=0.6)
ax.axvline(50, color=MUTED, ls="--", lw=1)
ax.text(50.5, 0, "50%", color=MUTED, fontsize=9, va="bottom")
ax.set_xlim(0, 100); ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title(f"{(miss > 50).sum()} columns are more than half empty")
ax.set_xlabel("Share of applicants with the value missing"); ax.grid(axis="y", visible=False)
save(fig, "missing_application", "Top 30 columns of application_train by share of missing values.")

In [ ]:
building = [c for c in app.columns if c.endswith(("_AVG", "_MODE", "_MEDI"))]
REPORT["n_building"] = len(building)
REPORT["building_miss"] = float(app[building].isna().mean().mean())
ext_miss = {c: float(app[c].isna().mean()) for c in ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]}
REPORT["ext_miss"] = ext_miss
note("Data quality", f"{REPORT['miss_cols']} of {app.shape[1]} application columns have gaps and "
     f"{REPORT['miss_over50']} are more than half empty. The worst group is the {len(building)} building-description "
     f"columns (_AVG/_MODE/_MEDI), {pct(REPORT['building_miss'])} empty on average. Among the external scores, "
     f"EXT_SOURCE_1 is {pct(ext_miss['EXT_SOURCE_1'])} missing, EXT_SOURCE_3 {pct(ext_miss['EXT_SOURCE_3'])} "
     f"and EXT_SOURCE_2 {pct(ext_miss['EXT_SOURCE_2'])}.")

hist_miss = pd.DataFrame({t: (data[t].isna().mean() * 100).round(1) for t in TABLES[1:]}).T
hist_miss = hist_miss.loc[:, (hist_miss > 0).any()]
print("Missing % in the history tables (only columns with gaps):")
display(hist_miss.T.dropna(how="all").sort_values(by=list(hist_miss.index), ascending=False).head(25))

### 4.2 Placeholder codes and anomalies
The data uses special values instead of blanks. They must be converted before any analysis or modelling, or they distort averages and models.

* **365243** in any `DAYS_*` column means *not applicable* (it would be 1,000 years in the future).
* **XNA / XAP** in text columns mean *not available / not applicable*.
* `DAYS_*` columns are days **before** the application, so they are negative.

In [ ]:
rows = []
for t, df in data.items():
    for c in [c for c in df.columns if c.startswith("DAYS")]:
        n = int((df[c] == 365243).sum())
        if n: rows.append({"table": t, "column": c, "value": "365243", "rows": n, "share %": round(n / len(df) * 100, 2)})
    for c in df.columns:
        if not pd.api.types.is_numeric_dtype(df[c]):
            for code_ in ["XNA", "XAP"]:
                n = int((df[c] == code_).sum())
                if n: rows.append({"table": t, "column": c, "value": code_, "rows": n, "share %": round(n / len(df) * 100, 2)})
placeholders = pd.DataFrame(rows).sort_values("rows", ascending=False)
REPORT["placeholders"] = placeholders
display(placeholders.head(25))

In [ ]:
emp_anom = app["DAYS_EMPLOYED"] == 365243
anom_rate, norm_rate = app.loc[emp_anom, "TARGET"].mean(), app.loc[~emp_anom, "TARGET"].mean()
anom_types = app.loc[emp_anom, "NAME_INCOME_TYPE"].value_counts(normalize=True).head(3)
REPORT.update(emp_anom=float(emp_anom.mean()), emp_anom_rate=float(anom_rate), emp_norm_rate=float(norm_rate))
print("Income type of applicants with DAYS_EMPLOYED = 365243:"); display(anom_types.round(3))
note("Data quality", f"{pct(emp_anom.mean())} of applicants have DAYS_EMPLOYED = 365243 (a placeholder). "
     f"{pct(anom_types.iloc[0])} of them are '{anom_types.index[0]}', so the code marks people without a job rather than "
     f"a data error. Their default rate ({pct(anom_rate)}) is {more_less(anom_rate, norm_rate)} the rest ({pct(norm_rate)}), "
     f"so replace the code with NaN but keep a flag column.")

In [ ]:
# Other anomalies worth knowing before modelling
checks = {
    "CODE_GENDER = XNA": int((app["CODE_GENDER"] == "XNA").sum()),
    "AMT_INCOME_TOTAL > 10 million": int((app["AMT_INCOME_TOTAL"] > 1e7).sum()),
    "AMT_ANNUITY missing": int(app["AMT_ANNUITY"].isna().sum()),
    "AMT_GOODS_PRICE missing": int(app["AMT_GOODS_PRICE"].isna().sum()),
    "CNT_CHILDREN >= 6": int((app["CNT_CHILDREN"] >= 6).sum()),
    "DAYS_EMPLOYED > 0 (future dates)": int((app["DAYS_EMPLOYED"] > 0).sum()),
    "installments: DAYS_ENTRY_PAYMENT missing": int(inst["DAYS_ENTRY_PAYMENT"].isna().sum()),
    "credit card: AMT_BALANCE < 0": int((cc["AMT_BALANCE"] < 0).sum()),
}
display(pd.Series(checks, name="rows").to_frame())
REPORT["anomalies"] = checks

### 4.3 Outliers in money columns

In [ ]:
money = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE"]
out = []
for c in money:
    s = app[c].dropna().astype(float)
    q1, q3 = s.quantile([.25, .75]); hi = q3 + 1.5 * (q3 - q1)
    out.append({"column": c, "median": s.median(), "99th pct": s.quantile(.99), "max": s.max(),
                "IQR upper fence": hi, "outliers %": round((s > hi).mean() * 100, 2)})
outliers = pd.DataFrame(out).set_index("column")
display(outliers.style.format("{:,.0f}", subset=["median", "99th pct", "max", "IQR upper fence"]))
inc = outliers.loc["AMT_INCOME_TOTAL"]
REPORT["income_max_over_p99"] = float(inc["max"] / inc["99th pct"])
note("Data quality", f"Money columns are strongly right-skewed. The highest income is "
     f"{REPORT['income_max_over_p99']:,.0f}× the 99th percentile, a clear outlier. Use log scales or ratios, "
     f"and cap extreme values rather than deleting rows.")

## 5. Target variable

In [ ]:
rate = app["TARGET"].mean(); counts = app["TARGET"].value_counts().sort_index()
REPORT.update(default_rate=float(rate), n_default=int(counts.get(1, 0)), n_repaid=int(counts.get(0, 0)))

fig, ax = plt.subplots(figsize=(7, 2.6))
ax.barh(["Defaulted (1)", "Repaid (0)"], [counts.get(1, 0), counts.get(0, 0)], color=[ORANGE, BLUE], height=0.55)
for y, v in enumerate([counts.get(1, 0), counts.get(0, 0)]):
    ax.text(v, y, f"  {v:,}  ({v/len(app):.1%})", va="center", color=INK2, fontsize=10)
ax.set_title(f"Only {rate:.1%} of applicants default: the classes are heavily imbalanced")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x/1000:,.0f}k")); ax.grid(axis="y", visible=False)
ax.set_xlim(0, counts.max() * 1.3)
save(fig, "target", "Number of applicants who repaid vs had payment difficulties.")
note("Target", f"{pct(rate)} of applicants defaulted ({REPORT['n_default']:,} of {len(app):,}), about 1 in "
     f"{1/rate:.0f}. A model that always predicts 'repaid' would be {pct(1-rate)} accurate and useless, so "
     f"evaluate with ROC-AUC, precision-recall and recall on defaulters, and use stratified splits and class weights.")

## 6. Applicant analysis (`application_train`)
### 6.1 Cleaning and derived features
Readable versions of the raw columns, plus the affordability ratios lenders actually look at.

In [ ]:
A = app.copy()
A["DAYS_EMPLOYED_ANOM"] = (A["DAYS_EMPLOYED"] == 365243).astype(int)
A["DAYS_EMPLOYED"] = A["DAYS_EMPLOYED"].replace(365243, np.nan)
A["AGE_YEARS"] = -A["DAYS_BIRTH"] / 365.25
A["YEARS_EMPLOYED"] = -A["DAYS_EMPLOYED"] / 365.25
A["CREDIT_INCOME_RATIO"] = A["AMT_CREDIT"] / A["AMT_INCOME_TOTAL"]
A["ANNUITY_INCOME_RATIO"] = A["AMT_ANNUITY"] / A["AMT_INCOME_TOTAL"]
A["CREDIT_TERM_YEARS"] = A["AMT_CREDIT"] / A["AMT_ANNUITY"]
A["GOODS_CREDIT_RATIO"] = A["AMT_GOODS_PRICE"] / A["AMT_CREDIT"]
A["EXT_SOURCE_MEAN"] = A[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].mean(axis=1)
display(A[["AGE_YEARS", "YEARS_EMPLOYED", "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "CREDIT_INCOME_RATIO",
           "ANNUITY_INCOME_RATIO", "CREDIT_TERM_YEARS", "EXT_SOURCE_MEAN"]].describe().T.round(2))

### 6.2 Distributions of key numeric features

In [ ]:
cols = [("AGE_YEARS", "Age (years)", False), ("YEARS_EMPLOYED", "Years employed", False),
        ("AMT_INCOME_TOTAL", "Annual income (log scale)", True), ("AMT_CREDIT", "Loan amount (log scale)", True),
        ("ANNUITY_INCOME_RATIO", "Annuity / income", False), ("CREDIT_TERM_YEARS", "Implied term (years)", False)]
fig, axes = plt.subplots(2, 3, figsize=(13, 6.5))
for ax, (c, label, logx) in zip(axes.flat, cols):
    s = A[c].dropna().astype(float)
    s = s[(s > 0)] if logx else s.clip(upper=s.quantile(.995))
    bins = np.logspace(np.log10(s.min()), np.log10(s.quantile(.999)), 40) if logx else 40
    ax.hist(s, bins=bins, color=BLUE, edgecolor=SURFACE, linewidth=0.6)
    if logx: ax.set_xscale("log")
    ax.set_title(label, fontsize=11); ax.set_yticks([]); ax.grid(axis="y", visible=False)
    ax.axvline(s.median(), color=INK2, lw=1, ls="--")
    ax.text(s.median(), ax.get_ylim()[1] * .95, f" median {s.median():,.2f}" if s.median() < 100 else f" median {s.median():,.0f}",
            color=INK2, fontsize=8.5, va="top")
fig.suptitle("Shape of the main numeric features", x=0.01, ha="left", fontsize=13, fontweight="bold")
fig.tight_layout()
save(fig, "distributions", "Distributions of age, employment length, income, loan amount and affordability ratios (dashed line = median).")
REPORT.update(age_median=float(A["AGE_YEARS"].median()), income_median=float(A["AMT_INCOME_TOTAL"].median()),
              credit_median=float(A["AMT_CREDIT"].median()), cir_median=float(A["CREDIT_INCOME_RATIO"].median()))
note("Applicant profile", f"The typical applicant is {REPORT['age_median']:.0f} years old, earns "
     f"{REPORT['income_median']:,.0f} a year and borrows {REPORT['credit_median']:,.0f}, about "
     f"{REPORT['cir_median']:.1f}× annual income.")

### 6.3 Default rate by applicant group
For each category: the share of applicants in that group who defaulted. The dashed line is the overall default rate, so bars to its right are riskier than average. Groups with fewer than 0.5% of applicants are folded into "Other" so tiny groups don't mislead.

In [ ]:
OVERALL = A["TARGET"].mean()

def group_rates(df, col, min_share=0.005, top=None):
    s = df[col].astype("object").fillna("Missing")
    share = s.value_counts(normalize=True)
    keep = share[share >= min_share].index
    if top: keep = share.loc[keep].head(top).index
    s = s.where(s.isin(keep), "Other")
    g = df.groupby(s, observed=True)["TARGET"].agg(rate="mean", n="size")
    g = g[~((g.index == "Other") & (g["n"] < min_share * len(df)))]   # drop a tiny leftover group
    return g.sort_values("rate")

def plot_rates(ax, g, title, xlabel="Default rate"):
    y = np.arange(len(g))
    ax.barh(y, g["rate"], color=BLUE, height=0.6)
    ax.set_yticks(y, [str(i)[:32] for i in g.index])
    ax.axvline(OVERALL, color=INK2, ls="--", lw=1)
    for yi, (r, n) in enumerate(zip(g["rate"], g["n"])):
        ax.text(r, yi, f" {r:.1%}  (n={n:,})", va="center", fontsize=8.5, color=INK2,
                bbox=dict(boxstyle="square,pad=0.1", fc=SURFACE, ec="none"))
    ax.set_xlim(0, g["rate"].max() * 1.45)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(1, decimals=0))
    ax.set_title(title, fontsize=11.5); ax.set_xlabel(xlabel); ax.grid(axis="y", visible=False)

CATS = [("CODE_GENDER", "Gender"), ("NAME_CONTRACT_TYPE", "Contract type"),
        ("NAME_EDUCATION_TYPE", "Education"), ("NAME_INCOME_TYPE", "Income type"),
        ("NAME_FAMILY_STATUS", "Family status"), ("NAME_HOUSING_TYPE", "Housing"),
        ("FLAG_OWN_CAR", "Owns a car"), ("FLAG_OWN_REALTY", "Owns property"),
        ("REGION_RATING_CLIENT_W_CITY", "Region rating (1 = best, 3 = worst)"), ("CNT_CHILDREN", "Number of children")]
CATS = [(c, t) for c, t in CATS if c in A.columns]
cat_tables = {c: group_rates(A, c) for c, _ in CATS}
nrow = int(np.ceil(len(CATS) / 2))
fig, axes = plt.subplots(nrow, 2, figsize=(13, 2.3 * nrow + 1))
for ax, (c, t) in zip(axes.flat, CATS):
    plot_rates(ax, cat_tables[c], t)
for ax in list(axes.flat)[len(CATS):]: ax.set_visible(False)
fig.suptitle(f"Default rate by applicant group (dashed line = overall {OVERALL:.1%})", x=0.01, ha="left",
             fontsize=13, fontweight="bold")
fig.tight_layout()
save(fig, "rates_by_category", "Default rate by gender, contract type, education, income type, family status, housing, car and property ownership, region rating and children.")

In [ ]:
# Occupation and organisation have many levels - show them on their own
fig, axes = plt.subplots(1, 2, figsize=(13, 7))
occ = group_rates(A, "OCCUPATION_TYPE"); org = group_rates(A, "ORGANIZATION_TYPE", top=20)
plot_rates(axes[0], occ, "Occupation"); plot_rates(axes[1], org, "Employer type (20 largest)")
fig.tight_layout()
save(fig, "rates_by_occupation", "Default rate by occupation and by employer type (20 largest).")
cat_tables["OCCUPATION_TYPE"] = occ; cat_tables["ORGANIZATION_TYPE"] = org

In [ ]:
# Turn the category tables into findings: biggest spread between groups with enough applicants
spread = []
for c, g in cat_tables.items():
    g2 = g[(g["n"] >= 0.01 * len(A)) & (g.index != "Other")]
    if len(g2) >= 2:
        spread.append({"feature": c, "lowest group": g2.index[0], "lowest rate": g2["rate"].iloc[0],
                       "highest group": g2.index[-1], "highest rate": g2["rate"].iloc[-1],
                       "ratio": g2["rate"].iloc[-1] / max(g2["rate"].iloc[0], 1e-9)})
spread = pd.DataFrame(spread).sort_values("ratio", ascending=False).set_index("feature")
REPORT["cat_spread"] = spread
display(spread.style.format({"lowest rate": "{:.1%}", "highest rate": "{:.1%}", "ratio": "{:.1f}×"}))
for f, r in spread.head(4).iterrows():
    note("Applicant groups", f"{f}: '{r['highest group']}' default at {pct(r['highest rate'])}, "
         f"{r['ratio']:.1f}× the rate of '{r['lowest group']}' ({pct(r['lowest rate'])}).")

### 6.4 Default rate across numeric features
Each numeric feature is split into ten equal-sized groups (deciles) and the default rate is shown per group. A steep line means a strong predictor.

In [ ]:
def decile_rates(df, col, q=10):
    s = df[col].astype(float)
    b = pd.qcut(s, q=q, duplicates="drop")
    g = df.groupby(b, observed=True)["TARGET"].agg(rate="mean", n="size")
    g["mid"] = df.groupby(b, observed=True)[col].median().astype(float).values
    return g

NUMS = [("EXT_SOURCE_1", "External score 1"), ("EXT_SOURCE_2", "External score 2"), ("EXT_SOURCE_3", "External score 3"),
        ("AGE_YEARS", "Age (years)"), ("YEARS_EMPLOYED", "Years employed"), ("CREDIT_INCOME_RATIO", "Loan / income"),
        ("ANNUITY_INCOME_RATIO", "Annuity / income"), ("CREDIT_TERM_YEARS", "Implied term (years)"),
        ("GOODS_CREDIT_RATIO", "Goods price / loan"), ("AMT_INCOME_TOTAL", "Income"),
        ("REGION_POPULATION_RELATIVE", "Region population density"), ("DAYS_LAST_PHONE_CHANGE", "Days since phone change")]
NUMS = [(c, t) for c, t in NUMS if c in A.columns]
num_tables = {c: decile_rates(A, c) for c, _ in NUMS}
ymax = max(g["rate"].max() for g in num_tables.values()) * 1.15
nrow = int(np.ceil(len(NUMS) / 3))
fig, axes = plt.subplots(nrow, 3, figsize=(13, 3 * nrow), sharey=True)
for ax, (c, t) in zip(axes.flat, NUMS):
    g = num_tables[c]
    ax.plot(g["mid"], g["rate"], color=BLUE, lw=2, marker="o", ms=4.5)
    ax.axhline(OVERALL, color=INK2, ls="--", lw=1)
    ax.set_title(t, fontsize=11); ax.set_ylim(0, ymax)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1, decimals=0))
    if c.startswith("AMT"): ax.set_xscale("log")
for ax in list(axes.flat)[len(NUMS):]: ax.set_visible(False)
fig.suptitle(f"Default rate by decile of each numeric feature (dashed line = overall {OVERALL:.1%})",
             x=0.01, ha="left", fontsize=13, fontweight="bold")
fig.tight_layout()
save(fig, "rates_by_decile", "Default rate across deciles of external scores, age, employment length, affordability ratios and other numeric features.")

trend = pd.DataFrame({c: {"lowest decile rate": g["rate"].iloc[0], "highest decile rate": g["rate"].iloc[-1],
                          "max/min ratio": g["rate"].max() / max(g["rate"].min(), 1e-9)} for c, g in num_tables.items()}).T
trend = trend.sort_values("max/min ratio", ascending=False)
REPORT["num_trend"] = trend
display(trend.style.format({"lowest decile rate": "{:.1%}", "highest decile rate": "{:.1%}", "max/min ratio": "{:.1f}×"}))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), sharey=True)
for ax, c in zip(axes, ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]):
    for tgt, col, lab in [(0, BLUE, "Repaid"), (1, ORANGE, "Defaulted")]:
        s = A.loc[A["TARGET"] == tgt, c].dropna().astype(float)
        ax.hist(s, bins=40, range=(0, 1), density=True, histtype="step", lw=2, color=col, label=lab)
    ax.set_title(c, fontsize=11); ax.set_yticks([]); ax.grid(axis="y", visible=False)
axes[0].legend(loc="upper left")
lower = all(A.loc[A.TARGET == 1, c].mean() < A.loc[A.TARGET == 0, c].mean() for c in ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"])
fig.suptitle("Defaulters have lower external scores" if lower else "External scores of defaulters vs non-defaulters", x=0.01, ha="left", fontsize=13, fontweight="bold")
fig.tight_layout()
save(fig, "ext_source_by_target", "Distribution of the three external credit scores for applicants who repaid vs defaulted.")

def decile_note(c, label, low_word, high_word):
    g = num_tables[c]; lo, hi = g["rate"].iloc[0], g["rate"].iloc[-1]
    ratio = max(lo, hi) / max(min(lo, hi), 1e-9)
    if ratio < 1.2:
        note("Numeric drivers", f"{label}: default rate barely changes, {pct(lo)} in the lowest decile vs {pct(hi)} "
             f"in the highest, so on its own it is a weak signal.")
    else:
        riskier = low_word if lo > hi else high_word
        note("Numeric drivers", f"{label}: default rate goes from {pct(lo)} in the lowest decile to {pct(hi)} in the "
             f"highest, so {riskier} applicants are riskier ({ratio:.1f}× difference).")

decile_note("EXT_SOURCE_2", "External score 2", "low-scoring", "high-scoring")
decile_note("EXT_SOURCE_3", "External score 3", "low-scoring", "high-scoring")
decile_note("AGE_YEARS", "Age", "younger", "older")
decile_note("YEARS_EMPLOYED", "Employment length", "recently employed", "long-employed")
decile_note("CREDIT_TERM_YEARS", "Implied loan term", "short-term", "long-term")

### 6.5 Correlation with the target

In [ ]:
RAW_DUPES = ("TARGET", "SK_ID_CURR", "DAYS_BIRTH", "DAYS_EMPLOYED")   # raw day counts duplicate AGE_YEARS / YEARS_EMPLOYED
num_cols = [c for c in A.columns if pd.api.types.is_numeric_dtype(A[c]) and c not in RAW_DUPES]
corr = A[num_cols].astype(float).corrwith(A["TARGET"].astype(float)).dropna().sort_values()
top = pd.concat([corr.head(10), corr.tail(10)])
REPORT["corr_top"] = top

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(top.index, top.values, color=[BLUE if v < 0 else ORANGE for v in top.values], height=0.6)
ax.axvline(0, color=AXIS, lw=1)
ax.set_title("Strongest linear links with default (Pearson correlation)")
ax.set_xlabel("← lower risk as the value rises      higher risk as the value rises →")
ax.grid(axis="y", visible=False)
save(fig, "correlation_target", "The 10 most negative and 10 most positive correlations of numeric application features with TARGET.")
note("Numeric drivers", f"The strongest single signals are {corr.index[0]} (r = {corr.iloc[0]:.2f}), "
     f"{corr.index[1]} ({corr.iloc[1]:.2f}) and {corr.index[2]} ({corr.iloc[2]:.2f}) on the protective side, and "
     f"{corr.index[-1]} (r = {corr.iloc[-1]:.2f}) and {corr.index[-2]} ({corr.iloc[-2]:.2f}) on the risk side. "
     + (f"Every correlation is weak (|r| ≤ {abs(corr).max():.2f}), so default depends on many features together, "
        "which suits a tree-based model." if abs(corr).max() < 0.3 else ""))

In [ ]:
# Redundancy: pairs of features that carry the same information
hm_cols = list(dict.fromkeys(list(corr.head(6).index) + list(corr.tail(6).index)))
cm = A[hm_cols].astype(float).corr()
fig, ax = plt.subplots(figsize=(9, 7.5))
im = ax.imshow(cm, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(hm_cols)), hm_cols, rotation=60, ha="right"); ax.set_yticks(range(len(hm_cols)), hm_cols)
for i in range(len(hm_cols)):
    for j in range(len(hm_cols)):
        v = cm.iloc[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7.5, color="white" if abs(v) > .6 else INK2)
ax.grid(False); fig.colorbar(im, ax=ax, shrink=.7)
ax.set_title("Correlation among the strongest features")
save(fig, "correlation_matrix", "Correlation matrix of the features most related to TARGET, used to spot redundant pairs.")

pairs = cm.where(np.triu(np.ones(cm.shape, bool), 1)).stack()
pairs = pairs[pairs.abs() > 0.8].sort_values(key=abs, ascending=False)
REPORT["redundant_pairs"] = pairs
if len(pairs):
    (a_, b_), v = pairs.index[0], pairs.iloc[0]
    note("Numeric drivers", f"{len(pairs)} feature pairs are highly correlated (|r| > 0.8), e.g. {a_} and {b_} "
         f"(r = {v:.2f}). Keep one of each pair for linear models.")

## 7. Credit history analysis
Each history table has many rows per applicant. To link it with default we **aggregate it to one row per applicant** (`SK_ID_CURR`) and join to `TARGET`, just like a SQL `GROUP BY` followed by a `LEFT JOIN`. Applicants with no rows in a table get a separate "none" group.

In [ ]:
T = A[["SK_ID_CURR", "TARGET"]]

def binned_rates(feat, col, bins, labels):
    # Default rate per bin of an aggregated feature; applicants without history form their own group.
    d = T.merge(feat[["SK_ID_CURR", col]], on="SK_ID_CURR", how="left")
    grp = pd.cut(d[col].astype(float), bins=bins, labels=labels, include_lowest=True).astype("object").fillna("No history")
    g = d.groupby(grp)["TARGET"].agg(rate="mean", n="size")
    order = [l for l in ["No history"] + list(labels) if l in g.index]
    return g.loc[order]

def plot_bins(ax, g, title, xlabel):
    x = np.arange(len(g))
    colors = [MUTED if i == "No history" else BLUE for i in g.index]
    ax.bar(x, g["rate"], color=colors, width=0.6)
    ax.set_xticks(x, g.index); ax.axhline(OVERALL, color=INK2, ls="--", lw=1)
    for xi, (r, n) in enumerate(zip(g["rate"], g["n"])):
        ax.text(xi, r + g["rate"].max() * .02, f"{r:.1%}\nn={n:,}", ha="center", va="bottom", fontsize=8, color=INK2,
                bbox=dict(boxstyle="square,pad=0.15", fc=SURFACE, ec="none"))
    ax.set_ylim(0, g["rate"].max() * 1.35); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1, decimals=0))
    ax.set_title(title, fontsize=11.5); ax.set_xlabel(xlabel); ax.grid(axis="x", visible=False)

HIST = {}   # aggregated features from every table, merged later

### 7.1 Credit bureau (`bureau` + `bureau_balance`)

In [ ]:
print("Loan status:"); display(bureau["CREDIT_ACTIVE"].value_counts(normalize=True).round(3).to_frame())
print("Loan type:");   display(bureau["CREDIT_TYPE"].value_counts(normalize=True).head(8).round(3).to_frame())

b = bureau.assign(active=(bureau["CREDIT_ACTIVE"] == "Active").astype(int),
                  overdue=(bureau["CREDIT_DAY_OVERDUE"] > 0).astype(int),
                  debt=bureau["AMT_CREDIT_SUM_DEBT"].astype(float).clip(lower=0).fillna(0))
bur = b.groupby("SK_ID_CURR").agg(BUR_LOANS=("SK_ID_BUREAU", "count"), BUR_ACTIVE=("active", "sum"),
                                  BUR_ANY_OVERDUE=("overdue", "max"), BUR_DEBT=("debt", "sum"),
                                  BUR_DAYS_CREDIT_MAX=("DAYS_CREDIT", "max")).reset_index()

# bureau_balance: worst monthly status ever per bureau loan, then per applicant
sev = {"C": 0, "X": 0, "0": 0, "1": 1, "2": 2, "3": 3, "4": 4, "5": 5}
bb_w = (bb.assign(sev=bb["STATUS"].map(sev).astype(float).fillna(0))
          .groupby("SK_ID_BUREAU")["sev"].max().rename("worst").reset_index()
          .merge(bureau[["SK_ID_BUREAU", "SK_ID_CURR"]], on="SK_ID_BUREAU")
          .groupby("SK_ID_CURR")["worst"].max().rename("BB_WORST_STATUS").reset_index())
bur = bur.merge(bb_w, on="SK_ID_CURR", how="left")
HIST["bureau"] = bur
print("STATUS in bureau_balance (C closed, X unknown, 0 on time, 1-5 = 1-30 ... 120+ days past due):")
display(bb["STATUS"].value_counts(normalize=True).round(3).to_frame())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
g1 = binned_rates(bur, "BUR_ACTIVE", [-.1, 0, 1, 2, 4, 1e9], ["0", "1", "2", "3-4", "5+"])
g2 = binned_rates(bur, "BUR_ANY_OVERDUE", [-.1, 0, 1], ["No", "Yes"])
g3 = binned_rates(bur, "BB_WORST_STATUS", [-.1, 0, 1, 5], ["Never late", "1-30 days", "31+ days"])
plot_bins(axes[0], g1, "Active loans at other lenders", "number of active loans")
plot_bins(axes[1], g2, "Currently overdue at another lender", "")
plot_bins(axes[2], g3, "Worst monthly bureau status", "")
fig.suptitle("Credit bureau history and default rate (grey = no bureau record)", x=0.01, ha="left", fontsize=13, fontweight="bold")
fig.tight_layout()
save(fig, "bureau", "Default rate by number of active bureau loans, current overdue status and worst historical bureau status.")
REPORT.update(bur_active=g1, bur_overdue=g2, bur_worst=g3)
if {"Yes", "No"} <= set(g2.index):
    note("Credit history", f"Applicants currently overdue at another lender default at {pct(g2.loc['Yes','rate'])}, "
         f"{more_less(g2.loc['Yes','rate'], g2.loc['No','rate'])} those who are not ({pct(g2.loc['No','rate'])}).")
g1h = g1.drop("No history", errors="ignore")
note("Credit history", f"Applicants with {g1h.index[-1]} active loans at other lenders default at {pct(g1h['rate'].iloc[-1])}, "
     f"{more_less(g1h['rate'].iloc[-1], g1h['rate'].iloc[0])} those with {g1h.index[0]} ({pct(g1h['rate'].iloc[0])})."
     + (f" Applicants with no bureau record default at {pct(g1.loc['No history','rate'])}." if "No history" in g1.index else ""))

### 7.2 Previous Home Credit applications (`previous_application`)

In [ ]:
print("Outcome of previous applications:")
display(prev["NAME_CONTRACT_STATUS"].value_counts(normalize=True).round(3).to_frame())
p = prev.assign(refused=(prev["NAME_CONTRACT_STATUS"] == "Refused").astype(int),
                approved=(prev["NAME_CONTRACT_STATUS"] == "Approved").astype(int))
pa = p.groupby("SK_ID_CURR").agg(PREV_APPS=("SK_ID_PREV", "count"), PREV_REFUSED=("refused", "sum"),
                                 PREV_APPROVED=("approved", "sum")).reset_index()
pa["PREV_REFUSED_SHARE"] = pa["PREV_REFUSED"] / pa["PREV_APPS"]
HIST["previous"] = pa

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
g1 = binned_rates(pa, "PREV_REFUSED", [-.1, 0, 1, 2, 1e9], ["0", "1", "2", "3+"])
g2 = binned_rates(pa, "PREV_APPS", [0, 1, 3, 6, 1e9], ["1", "2-3", "4-6", "7+"])
plot_bins(axes[0], g1, "Number of previously refused applications", "refusals")
plot_bins(axes[1], g2, "Number of previous applications", "applications")
fig.suptitle("Past applications at Home Credit and default rate", x=0.01, ha="left", fontsize=13, fontweight="bold")
fig.tight_layout()
save(fig, "previous_applications", "Default rate by number of refused and total previous Home Credit applications.")
REPORT["prev_refused"] = g1
note("Credit history", f"Applicants refused {g1.index[-1]} times before default at {pct(g1['rate'].iloc[-1])}, "
     f"{more_less(g1['rate'].iloc[-1], g1.loc['0','rate'])} those never refused ({pct(g1.loc['0','rate'])}). "
     f"{pct(prev['NAME_CONTRACT_STATUS'].eq('Refused').mean())} of all previous applications were refused.")

### 7.3 Repayment behaviour (`installments_payments`)

In [ ]:
i = inst.assign(days_late=(inst["DAYS_ENTRY_PAYMENT"] - inst["DAYS_INSTALMENT"]).astype(float),
                short=(inst["AMT_INSTALMENT"] - inst["AMT_PAYMENT"]).astype(float))
i["late"] = (i["days_late"] > 0).astype(int)
i["underpaid"] = (i["short"] > 1).astype(int)
print(f"Instalments paid late: {i['late'].mean():.1%}   underpaid: {i['underpaid'].mean():.1%}")
print("Days late, among late payments:"); display(i.loc[i["late"] == 1, "days_late"].describe().round(1).to_frame())
ins = i.groupby("SK_ID_CURR").agg(INS_COUNT=("late", "size"), INS_LATE_SHARE=("late", "mean"),
                                  INS_DAYS_LATE_MAX=("days_late", "max"),
                                  INS_UNDERPAID_SHARE=("underpaid", "mean")).reset_index()
HIST["installments"] = ins
REPORT.update(inst_late=float(i["late"].mean()), inst_under=float(i["underpaid"].mean()))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
g1 = binned_rates(ins, "INS_LATE_SHARE", [-.01, 0, .05, .15, 1], ["0%", "0-5%", "5-15%", "15%+"])
g2 = binned_rates(ins, "INS_UNDERPAID_SHARE", [-.01, 0, .05, .15, 1], ["0%", "0-5%", "5-15%", "15%+"])
plot_bins(axes[0], g1, "Share of past instalments paid late", "")
plot_bins(axes[1], g2, "Share of past instalments underpaid", "")
fig.suptitle("How applicants repaid earlier Home Credit loans", x=0.01, ha="left", fontsize=13, fontweight="bold")
fig.tight_layout()
save(fig, "installments", "Default rate by share of past instalments paid late and underpaid.")
REPORT["inst_late_bins"] = g1
if {"0%", "15%+"} <= set(g1.index):
    note("Credit history", f"Applicants who paid more than 15% of past instalments late default at {pct(g1.loc['15%+','rate'])}, "
         f"{more_less(g1.loc['15%+','rate'], g1.loc['0%','rate'])} those who always paid on time ({pct(g1.loc['0%','rate'])}).")

### 7.4 POS / cash loans and credit cards (`POS_CASH_balance`, `credit_card_balance`)

In [ ]:
ps = pos.assign(dpd=(pos["SK_DPD"] > 0).astype(int)).groupby("SK_ID_CURR").agg(
        POS_MONTHS=("MONTHS_BALANCE", "size"), POS_DPD_SHARE=("dpd", "mean")).reset_index()
c = cc.assign(util=(cc["AMT_BALANCE"].astype(float) / cc["AMT_CREDIT_LIMIT_ACTUAL"].replace(0, np.nan).astype(float)).clip(-1, 2),
              dpd=(cc["SK_DPD"] > 0).astype(int))
cs = c.groupby("SK_ID_CURR").agg(CC_UTIL_MEAN=("util", "mean"), CC_DPD_SHARE=("dpd", "mean")).reset_index()
HIST["pos"], HIST["cc"] = ps, cs
print(f"Months with any days past due - POS/cash: {pos['SK_DPD'].gt(0).mean():.1%}, credit card: {cc['SK_DPD'].gt(0).mean():.1%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
g1 = binned_rates(ps, "POS_DPD_SHARE", [-.01, 0, .1, 1], ["Never", "Up to 10%", "Over 10%"])
g2 = binned_rates(cs, "CC_UTIL_MEAN", [-1, .25, .5, .75, 2], ["<25%", "25-50%", "50-75%", "75%+"])
plot_bins(axes[0], g1, "POS/cash months with days past due", "")
plot_bins(axes[1], g2, "Average credit-card utilisation", "balance / limit")
fig.suptitle("Monthly loan and card behaviour and default rate", x=0.01, ha="left", fontsize=13, fontweight="bold")
fig.tight_layout()
save(fig, "pos_credit_card", "Default rate by share of POS/cash months past due and by average credit-card utilisation.")
REPORT["cc_util"] = g2
if "No history" in g2.index:
    REPORT["cc_coverage"] = 1 - g2.loc["No history", "n"] / len(T)
lo_u, hi_u = g2.drop("No history", errors="ignore")["rate"].iloc[[0, -1]]
note("Credit history", f"Applicants using over 75% of their card limit default at {pct(hi_u)}, {more_less(hi_u, lo_u)} "
     f"those using under 25% ({pct(lo_u)}). Only {pct(REPORT.get('cc_coverage', 0))} of applicants had a Home Credit card.")

## 8. Which signals matter most?
All aggregated history features are joined to the applicant table and ranked, with the application features, by how strongly they separate defaulters. **ROC-AUC of a single feature** is used: 0.5 means no signal, and the further from 0.5 the better. Unlike correlation, it ignores scale and catches non-linear but monotonic effects.

In [ ]:
from sklearn.metrics import roc_auc_score
F = A.copy()
for k, h in HIST.items():
    F = F.merge(h, on="SK_ID_CURR", how="left")
cand = [c for c in F.columns if pd.api.types.is_numeric_dtype(F[c]) and c not in RAW_DUPES
        and not c.startswith("FLAG_DOCUMENT") and F[c].notna().mean() > 0.1 and F[c].nunique() > 1]
res = []
for col in cand:
    m = F[col].notna()
    auc = roc_auc_score(F.loc[m, "TARGET"], F.loc[m, col].astype(float))
    res.append({"feature": col, "AUC": max(auc, 1 - auc), "direction": "higher = riskier" if auc > .5 else "higher = safer",
                "coverage": m.mean(), "source": next((k for k, h in HIST.items() if col in h.columns), "application")})
rank = pd.DataFrame(res).sort_values("AUC", ascending=False).set_index("feature")
REPORT["rank"] = rank
display(rank.head(20).style.format({"AUC": "{:.3f}", "coverage": "{:.0%}"}))

top = rank.head(15)[::-1]
src_col = {"application": BLUE, "bureau": ORANGE, "previous": AQUA, "installments": "#eda100", "pos": "#6250d6", "cc": "#e87ba4"}
fig, ax = plt.subplots(figsize=(9, 6.5))
ax.barh(top.index, top["AUC"] - .5, left=.5, color=[src_col[s] for s in top["source"]], height=0.6)
for y, (v, s) in enumerate(zip(top["AUC"], top["source"])):
    ax.text(v, y, f" {v:.3f}  {s}", va="center", fontsize=8.5, color=INK2)
ax.set_xlim(.5, top["AUC"].max() + .06); ax.grid(axis="y", visible=False)
ax.set_xlabel("Single-feature ROC-AUC (0.5 = no signal)")
ax.set_title(f"{rank.index[0]} is the strongest single signal")
save(fig, "feature_ranking", "The 15 strongest single features by ROC-AUC, coloured by source table.")
best_hist = rank[rank["source"] != "application"].head(3)
note("Feature ranking", f"The strongest single feature is {rank.index[0]} (AUC {rank['AUC'].iloc[0]:.3f}). "
     f"The best history-table features are " + ", ".join(f"{i} ({r['AUC']:.3f}, from {r['source']})" for i, r in best_hist.iterrows())
     + "." + (" No single feature reaches AUC 0.75 on its own, so a model must combine them."
              if rank["AUC"].iloc[0] < .75 else ""))

## 9. Summary of findings and recommendations

In [ ]:
for sec, items in FINDINGS.items():
    display(Markdown(f"**{sec}**\n\n" + "\n".join(f"- {t}" for t in items)))

### Recommendations for the modelling task
1. **Clean placeholders first:** replace 365243 with NaN (keep `DAYS_EMPLOYED_ANOM` as a flag), treat XNA/XAP as missing, fix `CODE_GENDER = XNA`.
2. **Handle imbalance:** stratified train/validation split, `class_weight` or `scale_pos_weight`, and judge models by ROC-AUC and precision-recall, never accuracy.
3. **Use gradient-boosted trees** (LightGBM / XGBoost). They handle missing values natively, capture non-linear effects such as age, and cope with weak individual signals.
4. **Engineer features from all tables:** the ratios built here plus counts, sums, means, maxima and "share late / refused / overdue" per applicant from every history table, with "no history" kept as its own signal.
5. **Drop or compress weak columns:** most `FLAG_DOCUMENT_*` columns and the duplicated building columns (_AVG/_MODE/_MEDI carry the same information).
6. **Check fairness:** gender and age show strong differences in default rate. Discuss with the business whether they may legally be used for credit decisions before including them.

## 10. Write the report
Creates `eda_outputs/Home_Loan_EDA_Report.html`: a stand-alone written report with every chart and finding above, computed from your data. Open it in a browser and use *Print → Save as PDF* to submit it.

In [ ]:
def img(path):
    return f'<img src="data:image/png;base64,{base64.b64encode(Path(path).read_bytes()).decode()}">'

def table(df, fmt=None):
    return (df.style.format(fmt) if fmt else df.style).to_html()

def fig_html(name):
    return "".join(f'<figure>{img(p)}<figcaption>Figure {k+1}. {cap}</figcaption></figure>'
                   for k, (p, cap) in enumerate(FIGS) if name in p.name)

def bullets(sec):
    return "<ul>" + "".join(f"<li>{t}</li>" for t in FINDINGS.get(sec, [])) + "</ul>"

R = REPORT
top3 = R["rank"].head(3)
exec_summary = [
    f"{pct(R['default_rate'])} of the {R['n_app']:,} applicants had payment difficulties: a heavily imbalanced target.",
    f"The strongest single predictors are {top3.index[0]} (AUC {top3['AUC'].iloc[0]:.3f}), {top3.index[1]} ({top3['AUC'].iloc[1]:.3f}) and {top3.index[2]} ({top3['AUC'].iloc[2]:.3f}).",
    f"The best credit-history signal is {best_hist.index[0]} from the {best_hist['source'].iloc[0]} table (AUC {best_hist['AUC'].iloc[0]:.3f}), so the history tables add information beyond the application.",
    f"Data quality needs work before modelling: {R['miss_over50']} columns are more than half empty and placeholder codes (365243, XNA, XAP) appear across the tables.",
    "Recommended approach: gradient-boosted trees on application features plus per-applicant aggregates of every history table, judged by ROC-AUC.",
]

html = f'''<!doctype html><html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Home Loan Default EDA Report</title>
<style>
 body{{font-family:-apple-system,Segoe UI,Roboto,Helvetica,Arial,sans-serif;color:#0b0b0b;background:#f9f9f7;margin:0;line-height:1.55}}
 main{{max-width:900px;margin:0 auto;padding:40px 24px 80px;background:#fcfcfb}}
 h1{{font-size:30px;margin:0 0 4px}} h2{{font-size:21px;margin-top:40px;border-bottom:1px solid #e1e0d9;padding-bottom:6px}}
 .sub{{color:#52514e;margin:0 0 24px}} li{{margin:6px 0}}
 figure{{margin:22px 0}} figure img{{max-width:100%;border:1px solid #e1e0d9;border-radius:6px}}
 figcaption{{color:#52514e;font-size:13px;margin-top:6px}}
 table{{border-collapse:collapse;font-size:12.5px;margin:12px 0;width:100%}} th,td{{border-bottom:1px solid #e1e0d9;padding:5px 8px;text-align:right}}
 th:first-child,td:first-child{{text-align:left}} thead th{{color:#52514e;font-weight:600}}
 .box{{background:#eef4fc;border-left:4px solid #2a78d6;padding:12px 18px;border-radius:4px}}
 @media print{{body{{background:#fff}} main{{padding:0}} figure{{break-inside:avoid}}}}
</style></head><body><main>
<h1>Home Loan Default Risk: Exploratory Data Analysis</h1>
<p class="sub">PRCP-1006 · Task 1: data analysis report · generated {datetime.date.today():%d %B %Y}</p>

<h2>1. Executive summary</h2><div class="box"><ul>{"".join(f"<li>{s}</li>" for s in exec_summary)}</ul></div>

<h2>2. Objective and data</h2>
<p>Home Credit wants to predict, at the time of application, whether a borrower will have payment difficulties
(TARGET = 1). The analysis uses seven related tables: the application itself, the applicant's loans at other lenders
(credit bureau) and their past applications, loans, cards and repayments with Home Credit.</p>
{table(R["overview"], {"rows": "{:,}", "unique keys": "{:,}", "rows per applicant": lambda v: "—" if pd.isna(v) else f"{v:.1f}", "missing cells %": "{:.1f}"})}
<p>Share of applicants with records in each history table:</p>{table(R["coverage"], "{:.1f}")}
{bullets("Data overview")}

<h2>3. Data quality</h2>
{bullets("Data quality")}
{fig_html("missing")}
<p>Placeholder codes found (largest first):</p>{table(R["placeholders"].head(12).set_index("table"), {"rows": "{:,}", "share %": "{:.2f}"})}

<h2>4. Target variable</h2>{fig_html("target")}{bullets("Target")}

<h2>5. Who defaults: applicant characteristics</h2>
{bullets("Applicant profile")}{fig_html("distributions")}
{fig_html("rates_by_category")}{fig_html("rates_by_occupation")}
{bullets("Applicant groups")}
{fig_html("rates_by_decile")}{fig_html("ext_source")}
{bullets("Numeric drivers")}{fig_html("correlation_target")}{fig_html("correlation_matrix")}

<h2>6. Credit history</h2>
{fig_html("bureau")}{fig_html("previous")}{fig_html("installments")}{fig_html("pos_credit")}
{bullets("Credit history")}

<h2>7. Strongest signals</h2>{fig_html("feature_ranking")}
{table(R["rank"].head(15)[["AUC", "direction", "coverage", "source"]], {"AUC": "{:.3f}", "coverage": "{:.0%}"})}
{bullets("Feature ranking")}

<h2>8. Recommendations for modelling</h2><ol>
<li><b>Clean placeholders:</b> replace 365243 with missing and keep a flag; treat XNA/XAP as missing.</li>
<li><b>Handle the imbalance:</b> stratified splits, class weights, and ROC-AUC / precision-recall instead of accuracy.</li>
<li><b>Model with gradient-boosted trees</b> (LightGBM / XGBoost), which handle missing values and non-linear effects.</li>
<li><b>Engineer features from every table:</b> affordability ratios plus per-applicant counts, sums, maxima and late / refused / overdue shares.</li>
<li><b>Remove redundant and empty columns,</b> e.g. duplicated building statistics and rarely used document flags.</li>
<li><b>Review fairness:</b> check whether gender and age may legally be used in credit decisions.</li>
</ol>
</main></body></html>'''

report_path = OUT_DIR / "Home_Loan_EDA_Report.html"
report_path.write_text(html, encoding="utf-8")
print(f"Report written: {report_path.resolve()}  ({report_path.stat().st_size/1e6:.1f} MB, {len(FIGS)} figures)")